In [30]:
from embedder import Embedder

embed = Embedder()

q1 = "How does approximate nearest neighbor search work?"

v = embed.encode(q1)

In [31]:
print(len(v))
print(v[0])

384
-0.02058203437252893


In [33]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [34]:
page = next(
    (
        doc for doc in documents
        if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"
    ),
    None
)

if page is None:
    print("Page not found")
else:
    dv = embed.encode(page["content"])
    cosine_similarity = v.dot(dv)
    print(cosine_similarity)

0.36107027225589694


In [37]:
import numpy as np

from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)


chunk_texts = [chunk["content"] for chunk in chunks]
dvChunks = embed.encode_batch(chunk_texts)

X = np.array(dvChunks)

scores = X.dot(v)

best_idx = np.argmax(scores)

print(chunks[best_idx]["filename"])
# print(chunks[best_idx]["content"])
print(scores[best_idx])

02-vector-search/lessons/07-sqlitesearch-vector.md
0.6489017718578813


In [38]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["content"])
vindex.fit(dvChunks, chunks)

q2 = "What metric do we use to evaluate a search engine?"
v2 = embed.encode(q2)

results = vindex.search(v2, num_results=1)
print(results[0]["filename"])


04-evaluation/lessons/05-search-metrics.md
